In [37]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# Load the credit card dataset
df = pd.read_csv('fraudTest.csv')

# Convert dates to datetime objects
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

# Create customer Age column
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,age
0,0,2020-06-21 12:14:25,2.291164e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,-80.9355,333497.0,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1.371817e+09,33.986391,-81.200714,0.0,52.0
1,1,2020-06-21 12:14:33,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,-110.4360,302.0,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1.371817e+09,39.450498,-109.960431,0.0,30.0
2,2,2020-06-21 12:14:53,3.598215e+15,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,-73.5365,34496.0,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1.371817e+09,40.495810,-74.196111,0.0,49.0
3,3,2020-06-21 12:15:15,3.591920e+15,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,-80.8191,54767.0,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1.371817e+09,28.812398,-80.883061,0.0,32.0
4,4,2020-06-21 12:15:17,3.526826e+15,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,-85.0170,1126.0,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1.371817e+09,44.959148,-85.884734,0.0,65.0


**Discretization and Binning**

In [38]:
# 1. Custom Equal-Interval Binning using pd.cut() for Age
age_bins = [18, 30, 50, 70, 100]
age_labels = ['Young', 'Middle-Aged', 'Senior', 'Elderly']

df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

print("Age Group Distribution:")
print(df['age_group'].value_counts())




Age Group Distribution:
age_group
Middle-Aged    83498
Senior         44585
Young          32941
Elderly        18995
Name: count, dtype: int64


In [39]:
# 2. Equal-Frequency Binning using pd.qcut() for Transaction Amount (amt)

df['amt_quartile'] = pd.qcut(df['amt'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])

print("\nAmount Quartile Distribution:")
print(df['amt_quartile'].value_counts())


Amount Quartile Distribution:
amt_quartile
Low          45642
Very High    45607
High         45602
Medium       45576
Name: count, dtype: int64


**Outlier Detection and Filtering**

In [40]:
# 1. Threshold Filtering (Find transactions exceeding $100)
high_value_txns = df[df['amt'] > 100]
print(f"Number of transactions over $100: {len(high_value_txns)}")

Number of transactions over $100: 32776


In [41]:
# 2. Statistical Outlier Detection (Z-score > 3 standard deviations)
mean_amt = df['amt'].mean()
std_amt = df['amt'].std()

outliers = df[np.abs(df['amt'] - mean_amt) > (3 * std_amt)]

print("\nDetected Outlier Transactions:")
print(outliers[['trans_date_trans_time', 'category', 'amt', 'is_fraud']].head())


Detected Outlier Transactions:
    trans_date_trans_time      category      amt  is_fraud
133   2020-06-21 12:55:19        travel   558.03       0.0
167   2020-06-21 13:08:46  shopping_net  1199.45       0.0
428   2020-06-21 14:38:09  shopping_pos  1881.53       0.0
464   2020-06-21 14:54:46      misc_pos   552.46       0.0
600   2020-06-21 15:37:37        travel   616.78       0.0


**Feature Scaling and Normalization**


In [42]:
numeric_cols = ['amt', 'city_pop', 'age']

In [43]:
# 1. Min-Max Scaling (Rescales values between 0 and 1)
min_max_scaler = MinMaxScaler()
df_min_max = pd.DataFrame(
    min_max_scaler.fit_transform(df[numeric_cols]),
    columns=[col + '_minmax' for col in numeric_cols]
)

In [44]:
# 2. Z-score Standardization (Mean = 0, Std Dev = 1)
standard_scaler = StandardScaler()
df_standard = pd.DataFrame(
    standard_scaler.fit_transform(df[numeric_cols]),
    columns=[col + '_standard' for col in numeric_cols]
)

In [45]:
robust_scaler = RobustScaler()
df_robust = pd.DataFrame(
    robust_scaler.fit_transform(df[numeric_cols]),
    columns=[col + '_robust' for col in numeric_cols]
)

In [46]:

scaled_summary = pd.concat([df[numeric_cols], df_min_max, df_standard, df_robust], axis=1)
scaled_summary.head()

,amt,city_pop,age,amt_minmax,city_pop_minmax,age_minmax,amt_standard,city_pop_standard,age_standard,amt_robust,city_pop_robust,age_robust
0,2.86,333497.0,52.0,0.000141,0.114727,0.4625,-0.449845,0.814552,0.325900,-0.606853,17.477249,0.333333
1,29.84,302.0,30.0,0.002193,0.000096,0.1875,-0.267325,-0.293302,-0.931444,-0.239253,-0.111170,-0.583333
2,41.28,34496.0,49.0,0.003064,0.011860,0.4250,-0.189933,-0.179609,0.154444,-0.083384,1.693834,0.208333
3,60.05,54767.0,32.0,0.004491,0.018834,0.2125,-0.062954,-0.112209,-0.817140,0.172355,2.763883,-0.500000
4,3.19,1126.0,65.0,0.000167,0.000379,0.6250,-0.447613,-0.290562,1.068877,-0.602357,-0.067673,0.875000
